In [142]:
import os
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [143]:
# ==========================================
# Configuration
# ==========================================

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = PROJECT_DIR / "plots"
REPORTS_DIR = PROJECT_DIR / "reports"
LOGS_DIR = PROJECT_DIR / "logs"

# Create folders automatically if they don't exist
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 512
CHUNK_OVERLAP = 100
TOP_K = 5

print(f"Project Directory : {PROJECT_DIR}")
print(f"Chunk Size        : {CHUNK_SIZE}")
print(f"Chunk Overlap     : {CHUNK_OVERLAP}")
print(f"Top K             : {TOP_K}")

Project Directory : c:\Users\premc\OneDrive\Desktop\Rag_project\django-react-rag-chatbot\experiments
Chunk Size        : 512
Chunk Overlap     : 100
Top K             : 5


In [144]:
def start_timer():
    return perf_counter()


def stop_timer(start_time):
    return round(perf_counter() - start_time, 4)


def print_section(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

In [145]:
print_section("Loading PDF")

pdf_path = DATA_DIR / "sample.pdf"

start = start_timer()

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

loading_time = stop_timer(start)

print(f"PDF Path      : {pdf_path}")
print(f"Total Pages   : {len(documents)}")
print(f"Loading Time  : {loading_time} sec")


Loading PDF
PDF Path      : c:\Users\premc\OneDrive\Desktop\Rag_project\django-react-rag-chatbot\experiments\data\sample.pdf
Total Pages   : 177
Loading Time  : 7.2482 sec


In [146]:
print_section("Chunking Documents")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

start = start_timer()

document_chunks = text_splitter.split_documents(documents)

chunking_time = stop_timer(start)

print(f"Total Chunks  : {len(document_chunks)}")
print(f"Chunking Time : {chunking_time} sec")

print("\nFirst Chunk Metadata")
print(document_chunks[0].metadata)

print("\nFirst Chunk Length")
print(len(document_chunks[0].page_content))

print("\nFirst Chunk Preview")
print("-" * 60)
print(document_chunks[0].page_content[:300])


Chunking Documents
Total Chunks  : 559
Chunking Time : 0.0178 sec

First Chunk Metadata
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-01-04T05:56:26+00:00', 'source': 'c:\\Users\\premc\\OneDrive\\Desktop\\Rag_project\\django-react-rag-chatbot\\experiments\\data\\sample.pdf', 'total_pages': 177, 'page': 0, 'page_label': '1'}

First Chunk Length
483

First Chunk Preview
------------------------------------------------------------
ARTIFICIAL INTELLIGENCE 
& 
 MACHINE LEARNING 
          Lecture Notes 
B.TECH 
(III YEAR – II SEM) 
(2022-2023) 
Prepared by:  
                        Ms.Anitha Patibandla, Associate Professor 
                  Dr.B.Jyothi, Professor 
                                                    Ms.K.Bhava


In [147]:
# ==========================================
# Load Embedding Model
# ==========================================

from sentence_transformers import SentenceTransformer

print_section("Loading Embedding Model")

MODEL_NAME = "BAAI/bge-base-en-v1.5"

start = start_timer()

embedding_model = SentenceTransformer(MODEL_NAME)

loading_time = stop_timer(start)

print(f"Model Name   : {MODEL_NAME}")
print(f"Loading Time : {loading_time} sec")


Loading Embedding Model


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4016.10it/s]


Model Name   : BAAI/bge-base-en-v1.5
Loading Time : 5.6648 sec


In [148]:
# ==========================================
# Generate Embeddings
# ==========================================

print_section("Generating Embeddings")

chunk_texts = [doc.page_content for doc in document_chunks]
chunk_metadata = [doc.metadata for doc in document_chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
embedding_time = stop_timer(start)

print(f"Total Chunks        : {len(document_chunks)}")
print(f"Embedding Shape     : {embeddings.shape}")
print(f"Embedding Dimension : {embeddings.shape[1]}")
print(f"Embedding Time      : {embedding_time} sec")

print("\nFirst Embedding (First 10 Values)")
print(embeddings[0][:10])


Generating Embeddings


Batches: 100%|██████████| 18/18 [01:35<00:00,  5.33s/it]

Total Chunks        : 559
Embedding Shape     : (559, 768)
Embedding Dimension : 768
Embedding Time      : 101.627 sec

First Embedding (First 10 Values)
[ 0.02140104 -0.00333658 -0.00868126 -0.00279707  0.06761923  0.03447152
  0.03383594 -0.00545447 -0.010889   -0.0048381 ]


In [149]:
# ==========================================
# Build FAISS Flat Index (Cosine Similarity)
# ==========================================

import faiss

print_section("Building FAISS Flat Index")

embedding_dimension = embeddings.shape[1]

start = start_timer()

# Create Flat Index using Inner Product (Cosine Similarity)
faiss_flat = faiss.IndexFlatIP(embedding_dimension)

# Add all embeddings to the index
faiss_flat.add(embeddings)

build_time = stop_timer(start)

print(f"Index Type          : {type(faiss_flat).__name__}")
print(f"Embedding Dimension : {embedding_dimension}")
print(f"Vectors Stored      : {faiss_flat.ntotal}")
print(f"Build Time          : {build_time:.4f} sec")


Building FAISS Flat Index
Index Type          : IndexFlatIP
Embedding Dimension : 768
Vectors Stored      : 559
Build Time          : 0.0105 sec


In [150]:
# ==========================================
# Generate Query Embedding
# ==========================================

print_section("Generating Query Embedding")

query = "What is Artifical intelligence?"

start = start_timer()

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_time = stop_timer(start)

print(f"Query              : {query}")
print(f"Embedding Shape    : {query_embedding.shape}")
print(f"Generation Time    : {query_time:.4f} sec")

print("\nFirst 10 Values")
print(query_embedding[0][:10])


Generating Query Embedding
Query              : What is Artifical intelligence?
Embedding Shape    : (1, 768)
Generation Time    : 0.1232 sec

First 10 Values
[ 0.01315913 -0.02500408  0.02021555 -0.04123097  0.05674537  0.01164882
  0.06479086 -0.00179914 -0.04226255 -0.00121918]


In [151]:
# ==========================================
# Search - FAISS Flat Index
# ==========================================

print_section("FAISS Flat Search")

query = "What is Artifical intelligence?"

# Generate query embedding
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Search
start = start_timer()

scores, indices = faiss_flat.search(
    query_embedding,
    TOP_K
)

search_time = stop_timer(start)

print(f"Query        : {query}")
print(f"Search Time  : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


FAISS Flat Search
Query        : What is Artifical intelligence?
Search Time  : 0.003100 sec

Top Results
----------------------------------------
Rank 1  | Score : 0.6796 | Chunk : 12
Rank 2  | Score : 0.6535 | Chunk : 28
Rank 3  | Score : 0.6514 | Chunk : 13
Rank 4  | Score : 0.6387 | Chunk : 29
Rank 5  | Score : 0.6177 | Chunk : 14


In [152]:
results = []

results.append({
    "Database": "FAISS",
    "Index": "Flat (IP)",
    "Build Time (s)": build_time,
    "Search Time (s)": search_time,
    "Average Similarity": round(scores[0].mean(), 4),
    "Top Similarity": round(scores[0][0], 4)
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796


In [153]:
# ==========================================
# Build FAISS IVF Index
# ==========================================

print_section("Building FAISS IVF Index")

dimension = embeddings.shape[1]

nlist = 10

quantizer = faiss.IndexFlatIP(dimension)

faiss_ivf = faiss.IndexIVFFlat(
    quantizer,
    dimension,
    nlist,
    faiss.METRIC_INNER_PRODUCT
)

start = start_timer()

faiss_ivf.train(embeddings)

faiss_ivf.add(embeddings)

build_time = stop_timer(start)

print(f"Index Type     : {type(faiss_ivf).__name__}")
print(f"Clusters       : {nlist}")
print(f"Vectors Stored : {faiss_ivf.ntotal}")
print(f"Build Time     : {build_time:.4f} sec")


Building FAISS IVF Index
Index Type     : IndexIVFFlat
Clusters       : 10
Vectors Stored : 559
Build Time     : 0.0320 sec


In [154]:
# ==========================================
# Search FAISS IVF Index
# ==========================================

print_section("FAISS IVF Search")

query = "What is Artifical intelligence?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Number of clusters to search
faiss_ivf.nprobe = 3

start = start_timer()

scores, indices = faiss_ivf.search(
    query_embedding,
    TOP_K
)

search_time = stop_timer(start)

print(f"Query        : {query}")
print(f"nprobe       : {faiss_ivf.nprobe}")
print(f"Search Time  : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


FAISS IVF Search
Query        : What is Artifical intelligence?
nprobe       : 3
Search Time  : 0.001500 sec

Top Results
----------------------------------------
Rank 1  | Score : 0.6796 | Chunk : 12
Rank 2  | Score : 0.6535 | Chunk : 28
Rank 3  | Score : 0.6514 | Chunk : 13
Rank 4  | Score : 0.6387 | Chunk : 29
Rank 5  | Score : 0.6177 | Chunk : 14


In [155]:
results.append({
    "Database": "FAISS",
    "Index": "IVF",
    "Build Time (s)": build_time,
    "Search Time (s)": search_time,
    "Average Similarity": round(scores[0].mean(), 4),
    "Top Similarity": round(scores[0][0], 4)
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796


In [156]:
# ==========================================
# Build FAISS HNSW Index
# ==========================================

print_section("Building FAISS HNSW Index")

dimension = embeddings.shape[1]
M = 32

start = start_timer()

faiss_hnsw = faiss.IndexHNSWFlat(dimension, M)

# HNSW search parameter
faiss_hnsw.hnsw.efConstruction = 40
faiss_hnsw.hnsw.efSearch = 32

faiss_hnsw.add(embeddings)

build_time = stop_timer(start)

print(f"Vectors Stored : {faiss_hnsw.ntotal}")
print(f"Build Time     : {build_time:.4f} sec")


Building FAISS HNSW Index
Vectors Stored : 559
Build Time     : 0.0150 sec


In [157]:
# ==========================================
# Search FAISS HNSW Index
# ==========================================

print_section("FAISS HNSW Search")

query = "What is Artifical intelligence?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

start = start_timer()

faiss.normalize_L2(query_embedding)

scores, indices = faiss_hnsw.search(
    query_embedding,
    TOP_K
)

search_time = stop_timer(start)

print(f"Query        : {query}")
print(f"Search Time  : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


FAISS HNSW Search
Query        : What is Artifical intelligence?
Search Time  : 0.001800 sec

Top Results
----------------------------------------
Rank 1  | Score : 0.6408 | Chunk : 12
Rank 2  | Score : 0.6930 | Chunk : 28
Rank 3  | Score : 0.6973 | Chunk : 13
Rank 4  | Score : 0.7226 | Chunk : 29
Rank 5  | Score : 0.7647 | Chunk : 14


In [158]:
results.append({
    "Database": "FAISS",
    "Index": "HNSW",
    "Build Time (s)": build_time,
    "Search Time (s)": search_time,
    "Average Similarity": round(scores[0].mean(), 4),
    "Top Similarity": round(scores[0][0], 4)
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408


In [159]:
# ==========================================
# Build ChromaDB Collection
# ==========================================

import chromadb
from chromadb.config import Settings

print_section("Building ChromaDB Collection")

start = start_timer()

chroma_client = chromadb.Client(
    Settings(anonymized_telemetry=False)
)

collection = chroma_client.get_or_create_collection(
    name="retrieval_lab"
)

collection.add(
    ids=[str(i) for i in range(len(chunk_texts))],
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=chunk_metadata
)

build_time = stop_timer(start)

print(f"Documents Stored : {collection.count()}")
print(f"Build Time       : {build_time:.4f} sec")


Building ChromaDB Collection
Documents Stored : 559
Build Time       : 0.3315 sec


In [160]:
# ==========================================
# Search ChromaDB
# ==========================================

print_section("ChromaDB Search")

query = "What is Artifical intelligence?"

query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

start = start_timer()

chroma_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=TOP_K
)

search_time = stop_timer(start)

print(f"Query        : {query}")
print(f"Search Time  : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, (doc_id, distance) in enumerate(
    zip(chroma_results["ids"][0], chroma_results["distances"][0]),
    start=1
):
    similarity = 1 - distance
    print(f"Rank {rank:<2} | Similarity : {similarity:.4f} | Chunk : {doc_id}")


ChromaDB Search
Query        : What is Artifical intelligence?
Search Time  : 0.023100 sec

Top Results
----------------------------------------
Rank 1  | Similarity : 0.3592 | Chunk : 12
Rank 2  | Similarity : 0.3070 | Chunk : 28
Rank 3  | Similarity : 0.3027 | Chunk : 13
Rank 4  | Similarity : 0.2774 | Chunk : 29
Rank 5  | Similarity : 0.2353 | Chunk : 14


In [161]:
results.append({
    "Database": "ChromaDB",
    "Index": "Default",
    "Build Time (s)": build_time,
    "Search Time (s)": search_time,
    "Average Similarity": round(
        1 - (sum(chroma_results["distances"][0]) / len(chroma_results["distances"][0])),
        4
    ),
    "Top Similarity": round(
        1 - chroma_results["distances"][0][0],
        4
    )
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592


In [162]:
# ==========================================
# Connect to Pinecone
# ==========================================

from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv()

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

print("Connected Successfully")

Connected Successfully


In [163]:
# ==========================================
# Create Pinecone Index
# ==========================================

from pinecone import ServerlessSpec

index_name = "retrieval-lab"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=embeddings.shape[1],
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index Ready")

Index Ready


In [164]:
# ==========================================
# Connect to Index
# ==========================================

index = pc.Index(index_name)

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=768, total_vector_count=559, metric='cosine', namespaces=1)


In [165]:
# ==========================================
# Upload Embeddings
# ==========================================

start = start_timer()

vectors = [
    (
        str(i),
        embeddings[i].tolist(),
        chunk_metadata[i]
    )
    for i in range(len(embeddings))
]

index.upsert(vectors=vectors)

build_time = stop_timer(start)

print(f"Vectors Uploaded : {len(vectors)}")
print(f"Build Time       : {build_time:.4f} sec")

Vectors Uploaded : 559
Build Time       : 5.6085 sec


In [166]:
# ==========================================
# Search Pinecone
# ==========================================

query = "What is Artificial intelligence?"

query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

start = start_timer()

pinecone_results = index.query(
    vector=query_embedding.tolist(),
    top_k=TOP_K,
    include_metadata=True
)

search_time = stop_timer(start)

print(f"Search Time : {search_time:.6f} sec")

for rank, match in enumerate(pinecone_results["matches"], start=1):
    print(
        f"Rank {rank} | "
        f"Score : {match['score']:.4f} | "
        f"Chunk : {match['id']}"
    )

Search Time : 0.292600 sec
Rank 1 | Score : 0.8339 | Chunk : 12
Rank 2 | Score : 0.7672 | Chunk : 13
Rank 3 | Score : 0.7278 | Chunk : 26
Rank 4 | Score : 0.7143 | Chunk : 14
Rank 5 | Score : 0.6786 | Chunk : 11


In [167]:
results.append({
    "Database": "Pinecone",
    "Index": "Cosine",
    "Build Time (s)": build_time,
    "Search Time (s)": search_time,
    "Average Similarity": round(
        sum(match["score"] for match in pinecone_results["matches"]) / len(pinecone_results["matches"]),
        4
    ),
    "Top Similarity": round(
        pinecone_results["matches"][0]["score"],
        4
    )
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339


In [168]:
# ==========================================
# Import BM25
# ==========================================

from rank_bm25 import BM25Okapi

In [169]:
# ==========================================
# Build BM25 Index
# ==========================================

print_section("Building BM25 Index")

start = start_timer()

tokenized_chunks = [
    chunk.lower().split()
    for chunk in chunk_texts
]

bm25 = BM25Okapi(tokenized_chunks)

build_time = stop_timer(start)

print(f"Documents Indexed : {len(tokenized_chunks)}")
print(f"Build Time        : {build_time:.4f} sec")


Building BM25 Index
Documents Indexed : 559
Build Time        : 0.0413 sec


In [170]:
# ==========================================
# BM25 Search
# ==========================================

print_section("BM25 Search")

query = "What is Artificial intelligence?"

query_tokens = query.lower().split()

start = start_timer()

scores = bm25.get_scores(query_tokens)

search_time = stop_timer(start)

top_indices = sorted(
    range(len(scores)),
    key=lambda i: scores[i],
    reverse=True
)[:TOP_K]

print(f"Query        : {query}")
print(f"Search Time  : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, idx in enumerate(top_indices, start=1):
    print(f"Rank {rank:<2} | Score : {scores[idx]:.4f} | Chunk : {idx}")


BM25 Search
Query        : What is Artificial intelligence?
Search Time  : 0.002800 sec

Top Results
----------------------------------------
Rank 1  | Score : 7.7046 | Chunk : 532
Rank 2  | Score : 7.0433 | Chunk : 262
Rank 3  | Score : 6.5465 | Chunk : 12
Rank 4  | Score : 6.1290 | Chunk : 140
Rank 5  | Score : 6.1194 | Chunk : 11


In [171]:
results.append({
    "Database": "BM25",
    "Index": "Lexical",
    "Build Time (s)": build_time,
    "Search Time (s)": search_time,
    "Average Similarity": round(
        sum(scores[idx] for idx in top_indices) / len(top_indices),
        4
    ),
    "Top Similarity": round(
        scores[top_indices[0]],
        4
    )
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046


In [172]:
# ==========================================
# Hybrid Search (BM25 + FAISS)
# ==========================================

print_section("Hybrid Search")

query = "What is Artificial intelligence?"

# ---------- BM25 ----------
query_tokens = query.lower().split()
bm25_scores = bm25.get_scores(query_tokens)

# ---------- FAISS ----------
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

_, faiss_indices = faiss_flat.search(
    query_embedding,
    TOP_K
)

# ---------- Combine ----------
hybrid_scores = {}

# BM25 Contribution
for idx, score in enumerate(bm25_scores):
    hybrid_scores[idx] = score

# FAISS Contribution
for rank, idx in enumerate(faiss_indices[0]):
    hybrid_scores[idx] = hybrid_scores.get(idx, 0) + (TOP_K - rank)

# Sort Results
top_results = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:TOP_K]

print("\nTop Results")
print("-" * 40)

for rank, (idx, score) in enumerate(top_results, start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


Hybrid Search

Top Results
----------------------------------------
Rank 1  | Score : 11.5465 | Chunk : 12
Rank 2  | Score : 7.7046 | Chunk : 532
Rank 3  | Score : 7.1194 | Chunk : 11
Rank 4  | Score : 7.0433 | Chunk : 262
Rank 5  | Score : 6.1290 | Chunk : 140


In [173]:
results.append({
    "Database": "Hybrid",
    "Index": "BM25 + FAISS",
    "Build Time (s)": 0,
    "Search Time (s)": search_time,
    "Average Similarity": round(
        sum(score for _, score in top_results) / len(top_results),
        4
    ),
    "Top Similarity": round(
        top_results[0][1],
        4
    )
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046
6,Hybrid,BM25 + FAISS,0.0000,0.0028,7.9086,11.5465


In [174]:
# ==========================================
# Load CrossEncoder
# ==========================================

from sentence_transformers import CrossEncoder

print_section("Loading CrossEncoder")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("CrossEncoder Loaded Successfully")


Loading CrossEncoder


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3625.72it/s]


CrossEncoder Loaded Successfully


In [175]:
# ==========================================
# Rerank Hybrid Results
# ==========================================

print_section("Reranking")

query = "What is Artificial intelligence?"

documents = [
    chunk_texts[idx]
    for idx, _ in top_results
]

pairs = [
    (query, doc)
    for doc in documents
]

start = start_timer()

rerank_scores = reranker.predict(pairs)

search_time = stop_timer(start)

reranked = sorted(
    zip(top_results, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

print(f"Search Time : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, ((idx, _), score) in enumerate(reranked, start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


Reranking
Search Time : 0.120000 sec

Top Results
----------------------------------------
Rank 1  | Score : 9.5820 | Chunk : 12
Rank 2  | Score : 3.6358 | Chunk : 11
Rank 3  | Score : -0.9955 | Chunk : 140
Rank 4  | Score : -4.1245 | Chunk : 262
Rank 5  | Score : -8.3622 | Chunk : 532


In [176]:
results.append({
    "Database": "Reranker",
    "Index": "CrossEncoder",
    "Build Time (s)": 0,
    "Search Time (s)": search_time,
    "Average Similarity": round(sum(rerank_scores) / len(rerank_scores), 4),
    "Top Similarity": round(max(rerank_scores), 4)
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046
6,Hybrid,BM25 + FAISS,0.0000,0.0028,7.9086,11.5465
7,Reranker,CrossEncoder,0.0000,0.1200,-0.0529,9.5820


In [178]:
# ==========================================
# ChromaDB Hybrid Search
# ==========================================

print_section("ChromaDB Hybrid Search")

query = "What is Artificial intelligence?"

# ---------- BM25 ----------
query_tokens = query.lower().split()
bm25_scores = bm25.get_scores(query_tokens)

# ---------- ChromaDB ----------
query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

start = start_timer()

chroma_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=TOP_K
)

search_time = stop_timer(start)

# ---------- Combine ----------
hybrid_scores = {}

# BM25 contribution
for idx, score in enumerate(bm25_scores):
    hybrid_scores[idx] = score

# Chroma contribution
for rank, doc_id in enumerate(chroma_results["ids"][0]):
    hybrid_scores[int(doc_id)] = hybrid_scores.get(int(doc_id), 0) + (TOP_K - rank)

# Sort
top_results = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:TOP_K]

print(f"Search Time : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, (idx, score) in enumerate(top_results, start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


ChromaDB Hybrid Search
Search Time : 0.005500 sec

Top Results
----------------------------------------
Rank 1  | Score : 11.5465 | Chunk : 12
Rank 2  | Score : 7.7046 | Chunk : 532
Rank 3  | Score : 7.1194 | Chunk : 11
Rank 4  | Score : 7.0433 | Chunk : 262
Rank 5  | Score : 6.1290 | Chunk : 140


In [179]:
results.append({
    "Database": "ChromaDB",
    "Index": "Hybrid (BM25 + Chroma)",
    "Build Time (s)": 0,
    "Search Time (s)": search_time,
    "Average Similarity": round(
        sum(score for _, score in top_results) / len(top_results),
        4
    ),
    "Top Similarity": round(
        top_results[0][1],
        4
    )
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046
6,Hybrid,BM25 + FAISS,0.0000,0.0028,7.9086,11.5465
7,Reranker,CrossEncoder,0.0000,0.1200,-0.0529,9.5820
8,ChromaDB,Hybrid (BM25 + Chroma),0.0000,0.0055,7.9086,11.5465


In [180]:
# ==========================================
# ChromaDB Reranking
# ==========================================

print_section("ChromaDB Reranking")

query = "What is Artificial intelligence?"

documents = [
    chunk_texts[idx]
    for idx, _ in top_results
]

pairs = [
    (query, doc)
    for doc in documents
]

start = start_timer()

rerank_scores = reranker.predict(pairs)

search_time = stop_timer(start)

reranked = sorted(
    zip(top_results, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

print(f"Search Time : {search_time:.6f} sec")

print("\nTop Results")
print("-" * 40)

for rank, ((idx, _), score) in enumerate(reranked, start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


ChromaDB Reranking
Search Time : 0.557600 sec

Top Results
----------------------------------------
Rank 1  | Score : 9.5820 | Chunk : 12
Rank 2  | Score : 3.6358 | Chunk : 11
Rank 3  | Score : -0.9955 | Chunk : 140
Rank 4  | Score : -4.1245 | Chunk : 262
Rank 5  | Score : -8.3622 | Chunk : 532


In [181]:
results.append({
    "Database": "ChromaDB",
    "Index": "CrossEncoder",
    "Build Time (s)": 0,
    "Search Time (s)": search_time,
    "Average Similarity": round(
        sum(rerank_scores) / len(rerank_scores),
        4
    ),
    "Top Similarity": round(
        max(rerank_scores),
        4
    )
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046
6,Hybrid,BM25 + FAISS,0.0000,0.0028,7.9086,11.5465
7,Reranker,CrossEncoder,0.0000,0.1200,-0.0529,9.5820
8,ChromaDB,Hybrid (BM25 + Chroma),0.0000,0.0055,7.9086,11.5465
9,ChromaDB,CrossEncoder,0.0000,0.5576,-0.0529,9.5820


In [182]:
# ==========================================
# ChromaDB + Hybrid Search + Reranking
# ==========================================

print_section("ChromaDB + Hybrid Search + Reranking")

query = "What is Artifical intelligence?"

# ---------- BM25 ----------
query_tokens = query.lower().split()
bm25_scores = bm25.get_scores(query_tokens)

# ---------- ChromaDB ----------
query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

start = start_timer()

chroma_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=TOP_K
)

# ---------- Hybrid ----------
hybrid_scores = {}

for idx, score in enumerate(bm25_scores):
    hybrid_scores[idx] = score

for rank, doc_id in enumerate(chroma_results["ids"][0]):
    hybrid_scores[int(doc_id)] = hybrid_scores.get(int(doc_id), 0) + (TOP_K - rank)

top_results = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:TOP_K]

# ---------- Reranking ----------
documents = [chunk_texts[idx] for idx, _ in top_results]

pairs = [(query, doc) for doc in documents]

rerank_scores = reranker.predict(pairs)

reranked = sorted(
    zip(top_results, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

search_time = stop_timer(start)

print(f"Search Time : {search_time:.6f} sec")

print("\nFinal Results")
print("-" * 40)

for rank, ((idx, _), score) in enumerate(reranked, start=1):
    print(
        f"Rank {rank:<2} | "
        f"Score : {score:.4f} | "
        f"Chunk : {idx}"
    )


ChromaDB + Hybrid Search + Reranking
Search Time : 0.424300 sec

Final Results
----------------------------------------
Rank 1  | Score : -0.3188 | Chunk : 12
Rank 2  | Score : -10.6124 | Chunk : 24
Rank 3  | Score : -10.9561 | Chunk : 497
Rank 4  | Score : -11.2517 | Chunk : 284
Rank 5  | Score : -11.4147 | Chunk : 540


In [183]:
results.append({
    "Database": "ChromaDB",
    "Index": "Hybrid + Reranking",
    "Build Time (s)": 0,
    "Search Time (s)": search_time,
    "Average Similarity": round(sum(rerank_scores) / len(rerank_scores), 4),
    "Top Similarity": round(max(rerank_scores), 4)
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046
6,Hybrid,BM25 + FAISS,0.0000,0.0028,7.9086,11.5465
7,Reranker,CrossEncoder,0.0000,0.1200,-0.0529,9.5820
8,ChromaDB,Hybrid (BM25 + Chroma),0.0000,0.0055,7.9086,11.5465
9,ChromaDB,CrossEncoder,0.0000,0.5576,-0.0529,9.5820


In [185]:
# ==========================================
# FAISS + Hybrid Search + Reranking
# ==========================================

print_section("FAISS + Hybrid Search + Reranking")

query = "What is Artificial intelligence?"

# ---------- BM25 ----------
query_tokens = query.lower().split()
bm25_scores = bm25.get_scores(query_tokens)

# ---------- FAISS ----------
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

start = start_timer()

faiss_scores, faiss_indices = faiss_flat.search(
    query_embedding,
    TOP_K
)

# ---------- Hybrid ----------
hybrid_scores = {}

# BM25 Scores
for idx, score in enumerate(bm25_scores):
    hybrid_scores[idx] = score

# FAISS Scores
for rank, idx in enumerate(faiss_indices[0]):
    hybrid_scores[idx] = hybrid_scores.get(idx, 0) + (TOP_K - rank)

# Top Hybrid Results
top_results = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:TOP_K]

# ---------- Reranking ----------
documents = [
    chunk_texts[idx]
    for idx, _ in top_results
]

pairs = [
    (query, doc)
    for doc in documents
]

rerank_scores = reranker.predict(pairs)

reranked = sorted(
    zip(top_results, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

search_time = stop_timer(start)

print(f"Search Time : {search_time:.6f} sec")

print("\nFinal Results")
print("-" * 40)

for rank, ((idx, _), score) in enumerate(reranked, start=1):
    print(f"Rank {rank:<2} | Score : {score:.4f} | Chunk : {idx}")


FAISS + Hybrid Search + Reranking
Search Time : 0.149900 sec

Final Results
----------------------------------------
Rank 1  | Score : 9.5820 | Chunk : 12
Rank 2  | Score : 3.6358 | Chunk : 11
Rank 3  | Score : -0.9955 | Chunk : 140
Rank 4  | Score : -4.1245 | Chunk : 262
Rank 5  | Score : -8.3622 | Chunk : 532


In [186]:
results.append({
    "Database": "FAISS",
    "Index": "Hybrid + Reranking",
    "Build Time (s)": 0,
    "Search Time (s)": search_time,
    "Average Similarity": round(sum(rerank_scores) / len(rerank_scores), 4),
    "Top Similarity": round(max(rerank_scores), 4)
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046
6,Hybrid,BM25 + FAISS,0.0000,0.0028,7.9086,11.5465
7,Reranker,CrossEncoder,0.0000,0.1200,-0.0529,9.5820
8,ChromaDB,Hybrid (BM25 + Chroma),0.0000,0.0055,7.9086,11.5465
9,ChromaDB,CrossEncoder,0.0000,0.5576,-0.0529,9.5820


In [187]:
# ==========================================
# Pinecone + Hybrid Search + Reranking
# ==========================================

print_section("Pinecone + Hybrid Search + Reranking")

query = "What is Artificial intelligence?"

# ---------- BM25 ----------
query_tokens = query.lower().split()
bm25_scores = bm25.get_scores(query_tokens)

# ---------- Pinecone ----------
query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

start = start_timer()

pinecone_results = index.query(
    vector=query_embedding.tolist(),
    top_k=TOP_K,
    include_metadata=True
)

# ---------- Hybrid ----------
hybrid_scores = {}

# BM25 Scores
for idx, score in enumerate(bm25_scores):
    hybrid_scores[idx] = score

# Pinecone Scores
for rank, match in enumerate(pinecone_results["matches"]):
    idx = int(match["id"])
    hybrid_scores[idx] = hybrid_scores.get(idx, 0) + (TOP_K - rank)

# Top Hybrid Results
top_results = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:TOP_K]

# ---------- Reranking ----------
documents = [
    chunk_texts[idx]
    for idx, _ in top_results
]

pairs = [
    (query, doc)
    for doc in documents
]

rerank_scores = reranker.predict(pairs)

reranked = sorted(
    zip(top_results, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

search_time = stop_timer(start)

print(f"Search Time : {search_time:.6f} sec")

print("\nFinal Results")
print("-" * 40)

for rank, ((idx, _), score) in enumerate(reranked, start=1):
    print(
        f"Rank {rank:<2} | "
        f"Score : {score:.4f} | "
        f"Chunk : {idx}"
    )


Pinecone + Hybrid Search + Reranking
Search Time : 1.522300 sec

Final Results
----------------------------------------
Rank 1  | Score : 9.5820 | Chunk : 12
Rank 2  | Score : 3.6358 | Chunk : 11
Rank 3  | Score : -0.9955 | Chunk : 140
Rank 4  | Score : -4.1245 | Chunk : 262
Rank 5  | Score : -8.3622 | Chunk : 532


In [188]:
results.append({
    "Database": "Pinecone",
    "Index": "Hybrid + Reranking",
    "Build Time (s)": 0,
    "Search Time (s)": search_time,
    "Average Similarity": round(
        sum(rerank_scores) / len(rerank_scores),
        4
    ),
    "Top Similarity": round(
        max(rerank_scores),
        4
    )
})

results_df = pd.DataFrame(results)

results_df

,Database,Index,Build Time (s),Search Time (s),Average Similarity,Top Similarity
0,FAISS,Flat (IP),0.0105,0.0031,0.6482,0.6796
1,FAISS,IVF,0.0320,0.0015,0.6482,0.6796
2,FAISS,HNSW,0.0150,0.0018,0.7037,0.6408
3,ChromaDB,Default,0.3315,0.0231,0.2963,0.3592
4,Pinecone,Cosine,5.6085,0.2926,0.7443,0.8339
5,BM25,Lexical,0.0413,0.0028,6.7086,7.7046
6,Hybrid,BM25 + FAISS,0.0000,0.0028,7.9086,11.5465
7,Reranker,CrossEncoder,0.0000,0.1200,-0.0529,9.5820
8,ChromaDB,Hybrid (BM25 + Chroma),0.0000,0.0055,7.9086,11.5465
9,ChromaDB,CrossEncoder,0.0000,0.5576,-0.0529,9.5820


In [189]:
# ==========================================
# Save Benchmark Results
# ==========================================

results_df.to_csv(
    "results/final_benchmark_results.csv",
    index=False
)

print("Benchmark results saved successfully!")

Benchmark results saved successfully!
